# IZQAR BAF — algorithm comparison on expanding month folds

train months [0..k], valid months 3/4/5. v1 and v2 feature sets.
Penultimate (month 6) and final test (month 7) are never touched.
Research-only; no artifact export.

In [ ]:
from pathlib import Path
import json, subprocess, sys

EXPECTED_REVISION = '97ebd1b02290a3bb90a3edc8ece1e559b78ca3fc'
root = Path('/kaggle/working/izqar_baf_algos')
if root.exists():
    raise FileExistsError('Use fresh session')
subprocess.run(['git', 'clone', '--branch', 'feat/baf-research', 'https://github.com/Night011/izqar.git', str(root)], check=True)
subprocess.run(['git', 'checkout', '--detach', EXPECTED_REVISION], cwd=root, check=True)
actual = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=root, text=True).strip()
assert actual == EXPECTED_REVISION, f'unexpected revision {actual}'
backend = root / 'backend'
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '-q',
                '-r', str(backend / 'requirements-ml.txt'), 'lightgbm', 'catboost'], check=True)
matches = list(Path('/kaggle/input').rglob('Base.csv'))
assert len(matches) == 1, f'expected one Base.csv, got {matches}'
data = matches[0]
print('input:', data, 'bytes:', data.stat().st_size, 'revision:', actual)

In [ ]:
import subprocess, sys, json
from pathlib import Path

subprocess.run([sys.executable, str(backend / 'scripts' / 'research_baf_algorithms.py'),
                '--data', str(data), '--output', '/kaggle/working/baf_algorithms.metrics.json'],
               cwd=backend, check=True)
report = json.loads(Path('/kaggle/working/baf_algorithms.metrics.json').read_text(encoding='utf-8'))
for key, r in report['results'].items():
    m = r['mean_metrics']
    print(f"{key:16s} mean PR-AUC={m['average_precision_pr_auc']:.6f} F1={m['f1']:.4f} FP/10k={m['false_alerts_per_10000_operations']:.1f}")
print('test_evaluation_performed: False')